# CLAUDETTE Cross Market Data Preparation

This notebook parses the **CLAUDETTE Cross Market** dataset into a normalized CSV for later fusion with **100 ToS** and **ToS;DR**.

## What this notebook does

1. Reads the pre-aligned `sentences/`, `labels/`, and `tags/` files for each platform.
2. Expands multi-tag lines (e.g. `cr2 ter2`) into one row per annotation.
3. Keeps CLAUDETTE's **native topic codes** (`j`, `law`, `ltd`, etc.) — no `lawgic_topics.json` mapping yet.
4. Maps CLAUDETTE fairness levels to Lawgic-style scores:
   - `1` clearly fair → `1` (good)
   - `2` potentially unfair → `0` (neutral)
   - `3` clearly unfair → `-1` (bad)
5. Writes `generated_files/claudette_cross_market/claudette_cross_market_clauses.csv`.
6. Reloads the CSV with **pandas** so you can inspect it interactively in the notebook.

## Output columns

| Column | Meaning |
| --- | --- |
| `quoted_text` | The annotated clause/sentence |
| `claudette_tag` | Full tag, e.g. `j3`, `law1` |
| `claudette_topic_code` | Topic prefix, e.g. `j`, `law` |
| `claudette_fairness_level` | Native CLAUDETTE level: 1, 2, or 3 |
| `mapped_score` | Converted score: 1, 0, or -1 |
| `binary_label` | Dataset binary flag from `labels/` (1 = unfair/neutral annotated) |

## 1. Setup and paths

We resolve paths relative to the project root. The raw dataset lives under `datasets/claudette_cross_market/`; the generated CSV goes to `generated_files/claudette_cross_market/`.

**Prerequisite:** run this notebook with your project Python environment (pandas is listed in `notebooks/requirements.txt`).

In [1]:
import re
from pathlib import Path

import pandas as pd

# Resolve project root whether the notebook is run from repo root or notebooks/claudette/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "claudette":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

# Raw CLAUDETTE Cross Market files (one file per platform in each subfolder)
DATASET_DIR = PROJECT_ROOT / "datasets" / "claudette_cross_market"
SENTENCES_DIR = DATASET_DIR / "sentences"
LABELS_DIR = DATASET_DIR / "labels"
TAGS_DIR = DATASET_DIR / "tags"

# Where we write the normalized CSV
OUTPUT_DIR = PROJECT_ROOT / "generated_files" / "claudette_cross_market"
OUTPUT_CSV = OUTPUT_DIR / "claudette_cross_market_clauses.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pandas display options — show more text when inspecting quoted clauses
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 20)

print(f"Dataset directory: {DATASET_DIR}")
print(f"Output CSV: {OUTPUT_CSV}")

Dataset directory: /Users/riki/Coding Projects/Thesis/lawgic/datasets/claudette_cross_market
Output CSV: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/claudette_cross_market/claudette_cross_market_clauses.csv


## 2. CLAUDETTE topic vocabulary and score mapping

CLAUDETTE tags follow the pattern `{topic_code}{fairness_level}` — for example `j3` is jurisdiction, clearly unfair.

The topic codes below come from the CLAUDETTE paper (Table 1). We keep them as-is for now; mapping to `lawgic_topics.json` happens in a later notebook.

In [2]:
# Human-readable names for each CLAUDETTE topic prefix
CLAUDETTE_TOPIC_LABELS = {
    "j": "Jurisdiction",
    "law": "Choice of law",
    "ltd": "Limitation of liability",
    "ch": "Unilateral change",
    "ter": "Unilateral termination",
    "use": "Contract by using",
    "cr": "Content removal",
    "a": "Arbitration",
    "pinc": "Privacy policy incorporated into terms",
}

# Native CLAUDETTE fairness wording (the digit appended to each tag)
CLAUDETTE_FAIRNESS_LABELS = {
    1: "clearly fair",
    2: "potentially unfair",
    3: "clearly unfair",
}

# Convert CLAUDETTE 1/2/3 → Lawgic-style 1/0/-1 for later fusion
CLAUDETTE_TO_LAWGIC_SCORE = {
    1: 1,
    2: 0,
    3: -1,
}

# Regex to split a tag like "j3" into topic_code="j" and fairness_level=3
TAG_PATTERN = re.compile(r"^(?P<topic_code>[a-z]+)(?P<fairness_level>[123])$")

# Final CSV column order
FIELDNAMES = [
    "source_dataset",
    "platform",
    "sentence_index",
    "quoted_text",
    "claudette_tag",
    "claudette_topic_code",
    "claudette_topic_label",
    "claudette_fairness_level",
    "claudette_fairness_label",
    "mapped_score",
    "binary_label",
]

## 3. Parsing helpers

Each platform has three parallel text files with **one line per sentence**, aligned by line number:

- `sentences/<platform>.txt` — the clause text
- `labels/<platform>.txt` — binary flag (`0` or `1`)
- `tags/<platform>.txt` — CLAUDETTE tag(s), or blank if unannotated

**Important:** we split files on `\n` only (not `splitlines()`), because some sentences contain Unicode line-separator characters that would otherwise break alignment.

In [3]:
def read_lines(path: Path) -> list[str]:
    """Read a platform file as a list of lines, preserving embedded Unicode separators."""
    text = path.read_text(encoding="utf-8").replace("\r\n", "\n").replace("\r", "\n")
    lines = text.split("\n")
    if lines and lines[-1] == "":
        lines = lines[:-1]
    return lines


def parse_claudette_tag(tag: str) -> dict:
    """Split a tag like 'j3' into topic metadata and mapped score fields."""
    match = TAG_PATTERN.match(tag)
    if not match:
        raise ValueError(f"Invalid CLAUDETTE tag: {tag!r}")

    topic_code = match.group("topic_code")
    fairness_level = int(match.group("fairness_level"))

    return {
        "claudette_tag": tag,
        "claudette_topic_code": topic_code,
        "claudette_topic_label": CLAUDETTE_TOPIC_LABELS.get(topic_code, topic_code),
        "claudette_fairness_level": fairness_level,
        "claudette_fairness_label": CLAUDETTE_FAIRNESS_LABELS[fairness_level],
        "mapped_score": CLAUDETTE_TO_LAWGIC_SCORE[fairness_level],
    }


def iter_platform_rows(platform: str) -> list[dict]:
    """Yield one dict per (sentence, tag) pair for a single platform."""
    sentences = read_lines(SENTENCES_DIR / f"{platform}.txt")
    labels = read_lines(LABELS_DIR / f"{platform}.txt")
    tags = read_lines(TAGS_DIR / f"{platform}.txt")

    # All three files must have the same number of lines for alignment to hold
    if not (len(sentences) == len(labels) == len(tags)):
        raise ValueError(
            f"Line-count mismatch for {platform}: "
            f"sentences={len(sentences)}, labels={len(labels)}, tags={len(tags)}"
        )

    rows = []
    for sentence_index, (sentence, binary_label, tag_line) in enumerate(
        zip(sentences, labels, tags), start=1
    ):
        tag_line = tag_line.strip()
        if not tag_line:
            continue  # skip unannotated sentences

        # A single sentence can carry multiple tags, e.g. "cr2 ter2"
        for tag in tag_line.split():
            rows.append(
                {
                    "source_dataset": "claudette_cross_market",
                    "platform": platform,
                    "sentence_index": sentence_index,
                    "quoted_text": sentence.strip(),
                    **parse_claudette_tag(tag),
                    "binary_label": int(binary_label),
                }
            )

    return rows

## 4. Parse all platforms into a DataFrame

We iterate over every `sentences/*.txt` file, collect annotation rows, and load them into a pandas DataFrame for easy inspection before writing to disk.

In [4]:
# One file per platform in sentences/
platforms = sorted(path.stem for path in SENTENCES_DIR.glob("*.txt"))

all_rows: list[dict] = []
for platform in platforms:
    all_rows.extend(iter_platform_rows(platform))

# Build the in-memory dataset — this is what we will export and later inspect
claudette_df = pd.DataFrame(all_rows, columns=FIELDNAMES)

print(f"Parsed {len(claudette_df):,} annotation rows from {len(platforms):,} platforms")
claudette_df.head()

Parsed 3,556 annotation rows from 142 platforms


,source_dataset,platform,sentence_index,quoted_text,claudette_tag,claudette_topic_code,claudette_topic_label,claudette_fairness_level,claudette_fairness_label,mapped_score,binary_label
0,claudette_cross_market,23andme,36,"By using the Service, you agree to these TOS.",use2,use,Contract by using,2,potentially unfair,0,1
1,claudette_cross_market,23andme,87,You therefore acknowledge and agree that the form and nature of the Services which 23andMe provides may change from ...,ch2,ch,Unilateral change,2,potentially unfair,0,1
2,claudette_cross_market,23andme,88,"As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) ...",ch2,ch,Unilateral change,2,potentially unfair,0,1
3,claudette_cross_market,23andme,88,"As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) ...",ter3,ter,Unilateral termination,3,clearly unfair,-1,1
4,claudette_cross_market,23andme,96,You acknowledge and agree that while 23andMe may not currently have set a fixed upper limit on the number of transmi...,ch2,ch,Unilateral change,2,potentially unfair,0,1


## 5. Export to CSV

Write the DataFrame to `generated_files/claudette_cross_market/claudette_cross_market_clauses.csv`. This file is the CLAUDETTE contribution to the later combined training dataset.

In [5]:
claudette_df.to_csv(OUTPUT_CSV, index=False)

print(f"Wrote {len(claudette_df):,} rows")
print(f"Platforms: {claudette_df['platform'].nunique():,}")
print(f"Saved to: {OUTPUT_CSV}")

Wrote 3,556 rows
Platforms: 142
Saved to: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/claudette_cross_market/claudette_cross_market_clauses.csv


## 6. Parse-time summaries

Quick distribution checks on the DataFrame we just built — useful to spot class imbalance or missing topics before moving on.

In [6]:
# Rows per CLAUDETTE tag (e.g. j1, j2, j3)
display(
    claudette_df["claudette_tag"]
    .value_counts()
    .sort_index()
    .rename("rows")
    .to_frame()
)

# Rows per topic family
display(
    claudette_df["claudette_topic_label"]
    .value_counts()
    .rename("rows")
    .to_frame()
)

# Fairness level distribution
display(
    claudette_df["claudette_fairness_label"]
    .value_counts()
    .rename("rows")
    .to_frame()
)

# Platforms with the most annotated rows
display(
    claudette_df["platform"]
    .value_counts()
    .head(20)
    .rename("rows")
    .to_frame()
)

,rows
claudette_tag,
a1,9
a2,85
a3,71
ch2,500
ch3,6
cr2,150
cr3,111
j1,38
j2,1


,rows
claudette_topic_label,
Limitation of liability,1072
Unilateral termination,624
Unilateral change,506
Contract by using,370
Content removal,261
Choice of law,225
Jurisdiction,218
Arbitration,165
Privacy policy incorporated into terms,115


,rows
claudette_fairness_label,
potentially unfair,2738
clearly unfair,637
clearly fair,181


,rows
platform,
Muse_Terms_of_Sale_Terms_of_Service_and_Limited_Warranty,79
Myspace,65
Skype,52
Weebly,50
WeChat,50
TikTok,50
LindenLab,50
GoFundMe,48
Endomondo,47


## 7. Validation

Sanity checks before we treat this CSV as ground truth. If any assertion fails, something went wrong in parsing or alignment.

In [7]:
required_columns = set(FIELDNAMES)
missing_columns = required_columns - set(claudette_df.columns)

assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"
assert len(claudette_df) > 0, "No CLAUDETTE rows were parsed"
assert claudette_df["quoted_text"].str.strip().ne("").all(), "Some rows have empty quoted text"
assert claudette_df["claudette_tag"].str.match(TAG_PATTERN).all(), "Some rows have invalid tags"
assert set(claudette_df["claudette_fairness_level"]).issubset({1, 2, 3})
assert set(claudette_df["mapped_score"]).issubset({-1, 0, 1})
assert set(claudette_df["binary_label"]).issubset({0, 1})
assert OUTPUT_CSV.exists(), f"Expected output CSV does not exist: {OUTPUT_CSV}"

print("Validation passed")

Validation passed


---

## 8. Inspect the exported CSV

Everything below reloads the CSV from disk (rather than reusing `claudette_df` in memory). This mirrors how you would load the file in a downstream notebook or training script.

Use these cells to explore the dataset interactively — no need to open Excel.

In [8]:
# Reload from disk to confirm the export round-trips correctly
claudette_csv = pd.read_csv(OUTPUT_CSV)

print(f"Shape: {claudette_csv.shape[0]:,} rows × {claudette_csv.shape[1]} columns")
print(f"File: {OUTPUT_CSV}")

claudette_csv.head(10)

Shape: 3,556 rows × 11 columns
File: /Users/riki/Coding Projects/Thesis/lawgic/generated_files/claudette_cross_market/claudette_cross_market_clauses.csv


,source_dataset,platform,sentence_index,quoted_text,claudette_tag,claudette_topic_code,claudette_topic_label,claudette_fairness_level,claudette_fairness_label,mapped_score,binary_label
0,claudette_cross_market,23andme,36,"By using the Service, you agree to these TOS.",use2,use,Contract by using,2,potentially unfair,0,1
1,claudette_cross_market,23andme,87,You therefore acknowledge and agree that the form and nature of the Services which 23andMe provides may change from ...,ch2,ch,Unilateral change,2,potentially unfair,0,1
2,claudette_cross_market,23andme,88,"As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) ...",ch2,ch,Unilateral change,2,potentially unfair,0,1
3,claudette_cross_market,23andme,88,"As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) ...",ter3,ter,Unilateral termination,3,clearly unfair,-1,1
4,claudette_cross_market,23andme,96,You acknowledge and agree that while 23andMe may not currently have set a fixed upper limit on the number of transmi...,ch2,ch,Unilateral change,2,potentially unfair,0,1
5,claudette_cross_market,23andme,169,In case of breach of any one of these promises 23andMe may suspend or terminate your account and refuse any and all ...,ch2,ch,Unilateral change,2,potentially unfair,0,1
6,claudette_cross_market,23andme,169,In case of breach of any one of these promises 23andMe may suspend or terminate your account and refuse any and all ...,ter2,ter,Unilateral termination,2,potentially unfair,0,1
7,claudette_cross_market,23andme,174,"If you provide any Registration Information that is untrue, inaccurate, not current, or incomplete, or if 23andMe ha...",ter2,ter,Unilateral termination,2,potentially unfair,0,1
8,claudette_cross_market,23andme,181,23andMe cannot and will not be liable for any loss or damage arising from your failure to comply with this Section.,ltd2,ltd,Limitation of liability,2,potentially unfair,0,1
9,claudette_cross_market,23andme,218,You acknowledge and agree that you are solely responsible for (and that 23andMe has no responsibility to you or to a...,ltd2,ltd,Limitation of liability,2,potentially unfair,0,1


### Column types and missing values

`info()` shows dtypes; `isna().sum()` flags any columns with nulls.

In [9]:
claudette_csv.info()

missing = claudette_csv.isna().sum()
if missing.any():
    display(missing[missing > 0].to_frame("missing_values"))
else:
    print("No missing values in any column.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3556 entries, 0 to 3555
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   source_dataset            3556 non-null   object
 1   platform                  3556 non-null   object
 2   sentence_index            3556 non-null   int64 
 3   quoted_text               3556 non-null   object
 4   claudette_tag             3556 non-null   object
 5   claudette_topic_code      3556 non-null   object
 6   claudette_topic_label     3556 non-null   object
 7   claudette_fairness_level  3556 non-null   int64 
 8   claudette_fairness_label  3556 non-null   object
 9   mapped_score              3556 non-null   int64 
 10  binary_label              3556 non-null   int64 
dtypes: int64(4), object(7)
memory usage: 305.7+ KB
No missing values in any column.


### Score and topic distributions

A crosstab shows how many rows fall into each `(topic, mapped_score)` combination.

In [10]:
# Mapped score counts: -1 (bad), 0 (neutral), 1 (good)
display(claudette_csv["mapped_score"].value_counts().sort_index().rename("rows").to_frame())

# Topic × score crosstab — which topics skew unfair vs fair?
score_by_topic = pd.crosstab(
    claudette_csv["claudette_topic_label"],
    claudette_csv["mapped_score"],
    margins=True,
)
score_by_topic.columns = ["bad (-1)", "neutral (0)", "good (1)", "total"]
display(score_by_topic)

,rows
mapped_score,
-1,637
0,2738
1,181


,bad (-1),neutral (0),good (1),total
claudette_topic_label,,,,
Arbitration,71,85,9,165
Choice of law,1,191,33,225
Content removal,111,150,0,261
Contract by using,0,370,0,370
Jurisdiction,179,1,38,218
Limitation of liability,23,948,101,1072
Privacy policy incorporated into terms,0,115,0,115
Unilateral change,6,500,0,506
Unilateral termination,246,378,0,624


### Sample clauses by topic

Pick a topic and inspect a few `quoted_text` examples at each fairness level. Change `TOPIC_TO_INSPECT` to explore other categories.

In [11]:
TOPIC_TO_INSPECT = "Jurisdiction"  # try: "Choice of law", "Limitation of liability", "Arbitration", etc.

topic_df = claudette_csv[claudette_csv["claudette_topic_label"] == TOPIC_TO_INSPECT]

for score in [1, 0, -1]:
    subset = topic_df[topic_df["mapped_score"] == score]
    print(f"\n{'='*60}")
    print(f"{TOPIC_TO_INSPECT} | mapped_score={score} | n={len(subset)}")
    print("=" * 60)

    sample_cols = [
        "platform",
        "claudette_tag",
        "claudette_fairness_label",
        "mapped_score",
        "quoted_text",
    ]
    display(subset[sample_cols].head(3))


Jurisdiction | mapped_score=1 | n=38


,platform,claudette_tag,claudette_fairness_label,mapped_score,quoted_text
128,Airbnb,j1,clearly fair,1,Judicial proceedings that you are able to bring against us arising from or in connection with these Terms may only b...
129,Airbnb,j1,clearly fair,1,"If Airbnb wishes to enforce any of its rights against you as a consumer, we may do so only in the courts of the juri..."
207,Ava,j1,clearly fair,1,"If you reside in North America, this Agreement will be governed by and in accordance with the laws of the State of C..."



Jurisdiction | mapped_score=0 | n=1


,platform,claudette_tag,claudette_fairness_label,mapped_score,quoted_text
2392,Supercell,j2,potentially unfair,0,You agree that any claim or dispute you may have against Supercell must be resolved exclusively by a court located i...



Jurisdiction | mapped_score=-1 | n=179


,platform,claudette_tag,claudette_fairness_label,mapped_score,quoted_text
33,23andme,j3,clearly unfair,-1,<J3>These TOS shall be governed by the laws of England and Wales we each agree that any dispute that goes to court w...
79,Academia,j3,clearly unfair,-1,"The exclusive jurisdiction and venue of any IP Protection Action or, if you timely provide Academia.edu with an Arbi..."
103,Ada,j3,clearly unfair,-1,"The courts of Germany have the exclusive jurisdiction to settle any disputes arising in connection with, or as a res..."


### Filter by platform

Inspect all annotations for a single service. Useful when comparing CLAUDETTE labels against the original ToS text.

In [12]:
PLATFORM_TO_INSPECT = "Dropbox"  # change to any platform name from claudette_csv["platform"].unique()

platform_df = claudette_csv[claudette_csv["platform"] == PLATFORM_TO_INSPECT].sort_values(
    ["sentence_index", "claudette_tag"]
)

print(f"{PLATFORM_TO_INSPECT}: {len(platform_df)} annotated rows")

display(
    platform_df[
        [
            "sentence_index",
            "claudette_tag",
            "claudette_topic_label",
            "mapped_score",
            "quoted_text",
        ]
    ]
)

Dropbox: 25 annotated rows


,sentence_index,claudette_tag,claudette_topic_label,mapped_score,quoted_text
475,7,pinc2,Privacy policy incorporated into terms,0,"By using our Services, you're agreeing to be bound by these Terms, our Privacy Policy and Acceptable Use Policy."
474,7,use2,Contract by using,0,"By using our Services, you're agreeing to be bound by these Terms, our Privacy Policy and Acceptable Use Policy."
476,48,cr2,Content removal,0,We reserve the right to delete or disable content alleged to be infringing and terminate accounts of repeat infringers.
477,48,ter2,Unilateral termination,0,We reserve the right to delete or disable content alleged to be infringing and terminate accounts of repeat infringers.
478,68,ch2,Unilateral change,0,"If you don't pay for your Paid Account on time, we reserve the right to suspend it or reduce your storage to free sp..."
...,...,...,...,...,...
494,133,j3,Jurisdiction,-1,"If the agreement to arbitrate is found not to apply to you or your claim, you agree to the exclusive jurisdiction of..."
495,139,law2,Choice of law,0,These Terms will be governed by California law except for its conflicts of laws principles.
496,140,law1,Choice of law,1,"However, some countries (including those in the European Union) have laws that require agreements to be governed by ..."
497,151,ch2,Unilateral change,0,"We may revise these Terms from time to time to better reflect: (a) changes to the law, (b) new regulatory requiremen..."


### Multi-tag sentences

Some sentences carry more than one CLAUDETTE tag on the same line (e.g. `ch2 ter3`). These rows share the same `quoted_text` but differ in `claudette_tag`.

In [13]:
# Group by (platform, sentence_index) and find sentences with multiple annotation rows
multi_tag_keys = (
    claudette_csv.groupby(["platform", "sentence_index"])
    .size()
    .reset_index(name="tag_count")
    .query("tag_count > 1")
)

print(f"Sentences with multiple tags: {len(multi_tag_keys):,}")

# Show a few examples
example_keys = multi_tag_keys.head(5)[["platform", "sentence_index"]]
for _, row in example_keys.iterrows():
    mask = (
        (claudette_csv["platform"] == row["platform"])
        & (claudette_csv["sentence_index"] == row["sentence_index"])
    )
    print(f"\n--- {row['platform']} | sentence {row['sentence_index']} ---")
    display(
        claudette_csv.loc[mask, ["claudette_tag", "mapped_score", "quoted_text"]]
    )

Sentences with multiple tags: 310

--- 23andme | sentence 88 ---


,claudette_tag,mapped_score,quoted_text
2,ch2,0,"As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) ..."
3,ter3,-1,"As part of this continuing innovation, you acknowledge and agree that 23andMe may stop (permanently or temporarily) ..."



--- 23andme | sentence 169 ---


,claudette_tag,mapped_score,quoted_text
5,ch2,0,In case of breach of any one of these promises 23andMe may suspend or terminate your account and refuse any and all ...
6,ter2,0,In case of breach of any one of these promises 23andMe may suspend or terminate your account and refuse any and all ...



--- 23andme | sentence 273 ---


,claudette_tag,mapped_score,quoted_text
16,ch2,0,"23andMe may at any time and from time to time to modify or discontinue, temporarily or permanently, the Services (or..."
17,ter2,0,"23andMe may at any time and from time to time to modify or discontinue, temporarily or permanently, the Services (or..."



--- 23andme | sentence 359 ---


,claudette_tag,mapped_score,quoted_text
33,j3,-1,<J3>These TOS shall be governed by the laws of England and Wales we each agree that any dispute that goes to court w...
34,law2,0,<J3>These TOS shall be governed by the laws of England and Wales we each agree that any dispute that goes to court w...



--- 9gag | sentence 12 ---


,claudette_tag,mapped_score,quoted_text
36,ch2,0,"9GAG, Inc may change, suspend or discontinue the Services at any time, including the availability of any feature, da..."
37,ter3,-1,"9GAG, Inc may change, suspend or discontinue the Services at any time, including the availability of any feature, da..."
